In [1]:
import csv
import logging
import time
from pathlib import Path
from urllib.parse import urlparse

import undetected_chromedriver as uc
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

BASE_DIR = Path('/home/kongla/Documents/GitHub/Real-estate-Scraping/web-scraping/dotproperty')
START_URL = "https://www.dotproperty.co.th/properties-for-sale/bangkok?types=condos%2Cwhole-building%2Cwarehouses%2Cretail-spaces%2Ccommercial-property%2Cland%2Cvillas%2Cpenthouses%2Capartments%2Chouses%2Ctownhouses%2Cprivate-island%2Coffices%2Cshophouse%2Chotels%2Cserviced-apartments"
OUTPUT_CSV_FILE = BASE_DIR / 'dotproperty_listing_urls.csv'

TARGET_PAGES = 5
PAGE_TIMEOUT = 15

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def setup_optimized_driver() -> uc.Chrome:
    options = uc.ChromeOptions()
    options.page_load_strategy = 'eager'

    prefs = {
        "profile.managed_default_content_settings.images": 2,
        "profile.managed_default_content_settings.stylesheets": 2,
        "profile.default_content_setting_values.notifications": 2,
        "profile.managed_default_content_settings.fonts": 2,
    }
    options.add_experimental_option("prefs", prefs)

    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-extensions")

    return uc.Chrome(options=options, version_main=145)

def extract_links(driver: uc.Chrome, base_url: str) -> list[str]:
    js = """
    return [...new Set(
        Array.from(document.querySelectorAll('a[href^="/ads/"]'))
        .map(a => a.getAttribute('href'))
        .filter(Boolean)
    )];
    """
    hrefs = driver.execute_script(js)
    return [f"{base_url}{h}" if h.startswith("/") else h for h in hrefs]

def main():
    BASE_DIR.mkdir(parents=True, exist_ok=True)

    driver = setup_optimized_driver()
    wait = WebDriverWait(driver, PAGE_TIMEOUT)
    parsed_url = urlparse(START_URL)
    base_url = f"{parsed_url.scheme}://{parsed_url.netloc}"

    all_urls = set()

    try:
        logging.info(f"Starting scrape at: {START_URL}")
        driver.get(START_URL)
        
        current_page = 1
        
        while current_page <= TARGET_PAGES:
            try:
                wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'a[href^="/ads/"]')))
            except TimeoutException:
                logging.warning(f"Timeout waiting for elements on page {current_page}. Stopping.")
                break
                
            current_links = extract_links(driver, base_url)
            all_urls.update(current_links)
            
            logging.info(f"Page {current_page}: Collected {len(current_links)} URLs. Total: {len(all_urls)}")

            if current_page >= TARGET_PAGES:
                break

            try:
                next_button = driver.find_element(By.CSS_SELECTOR, 'a[rel="next"][aria-label="Pagination"]')
                next_url = next_button.get_attribute("href")
                driver.get(next_url)
                current_page += 1
                time.sleep(2)
            except NoSuchElementException:
                logging.info("Next page button not found. Reached the end.")
                break

    except KeyboardInterrupt:
        logging.info("Scraping manually interrupted by user.")
    except Exception as e:
        logging.error(f"Unexpected error: {e}")
    finally:
        driver.quit()

    final_urls = sorted(list(all_urls))

    with OUTPUT_CSV_FILE.open('w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['ListingURL'])
        for u in final_urls:
            writer.writerow([u])

    logging.info(f"Successfully saved {len(final_urls)} URLs to {OUTPUT_CSV_FILE}")

if __name__ == "__main__":
    main()

2026-03-29 16:46:23,470 - INFO - patching driver executable /home/kongla/.local/share/undetected_chromedriver/undetected_chromedriver
2026-03-29 16:46:24,280 - INFO - Starting scrape at: https://www.dotproperty.co.th/properties-for-sale/bangkok?types=condos%2Cwhole-building%2Cwarehouses%2Cretail-spaces%2Ccommercial-property%2Cland%2Cvillas%2Cpenthouses%2Capartments%2Chouses%2Ctownhouses%2Cprivate-island%2Coffices%2Cshophouse%2Chotels%2Cserviced-apartments
2026-03-29 16:46:26,200 - INFO - Page 1: Collected 30 URLs. Total: 30
2026-03-29 16:46:30,325 - INFO - Page 2: Collected 30 URLs. Total: 60
2026-03-29 16:46:33,748 - INFO - Page 3: Collected 30 URLs. Total: 90
2026-03-29 16:46:37,915 - INFO - Page 4: Collected 30 URLs. Total: 120
2026-03-29 16:46:41,578 - INFO - Page 5: Collected 30 URLs. Total: 150
2026-03-29 16:46:41,690 - INFO - Successfully saved 150 URLs to /home/kongla/Documents/GitHub/Real-estate-Scraping/web-scraping/dotproperty/dotproperty_listing_urls.csv


In [2]:
import csv
import logging
import time
from pathlib import Path
from urllib.parse import urlparse

import undetected_chromedriver as uc
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

BASE_DIR = Path('/home/kongla/Documents/GitHub/Real-estate-Scraping/web-scraping/dotproperty')
OUTPUT_CSV_FILE = BASE_DIR / 'dotproperty_full_details.csv'
INPUT_CSV = BASE_DIR / 'dotproperty_listing_urls.csv'
PAGE_TIMEOUT = 15

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def setup_optimized_driver() -> uc.Chrome:
    options = uc.ChromeOptions()
    options.page_load_strategy = 'eager'

    prefs = {
        "profile.managed_default_content_settings.images": 2,
        "profile.managed_default_content_settings.stylesheets": 2,
        "profile.default_content_setting_values.notifications": 2,
        "profile.managed_default_content_settings.fonts": 2,
    }
    options.add_experimental_option("prefs", prefs)

    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-extensions")

    return uc.Chrome(options=options, version_main=145)

def extract_content(driver: uc.Chrome) -> str:
    data = []
    
    try:
        title = driver.find_element(By.TAG_NAME, 'h1').text.strip()
        data.extend(["--- Listing Title ---", title])
    except NoSuchElementException:
        pass

    try:
        price = driver.find_element(By.CSS_SELECTOR, '.text-2xl.font-medium').text.strip()
        data.extend(["\n--- Price ---", price])
    except NoSuchElementException:
        pass

    try:
        room_info = driver.find_element(By.CSS_SELECTOR, 'div[data-testid="unit-search-result-item-details"]').text.strip()
        data.extend(["\n--- Room Info ---", room_info])
    except NoSuchElementException:
        pass

    try:
        address = driver.find_element(By.CSS_SELECTOR, 'address[data-testid="unit-search-result-item-address"]').text.strip()
        data.extend(["\n--- Address ---", address])
    except NoSuchElementException:
        pass

    try:
        desc = driver.find_element(By.CSS_SELECTOR, '.line-clamp-3.font-light.text-gray-700').text.strip()
        if desc:
            data.extend(["\n--- Description ---", desc])
    except NoSuchElementException:
        pass
        
    try:
        basic_info_elements = driver.find_elements(By.CSS_SELECTOR, '#basic-information + div .relative.flex.flex-col')
        if basic_info_elements:
            data.append("\n--- Basic Information ---")
            for item in basic_info_elements:
                try:
                    label = item.find_elements(By.TAG_NAME, 'div')[1].text.strip()
                    value = item.find_elements(By.TAG_NAME, 'div')[2].text.strip()
                    data.append(f"{label}: {value}")
                except Exception:
                    continue
    except NoSuchElementException:
        pass
        
    try:
        facilities_elements = driver.find_elements(By.CSS_SELECTOR, '#project-features + div .flex.items-start')
        if facilities_elements:
            data.append("\n--- Facilities ---")
            facilities = [f.text.strip() for f in facilities_elements]
            data.append(", ".join(facilities))
    except NoSuchElementException:
        pass

    return "\n".join(data)

def main():
    BASE_DIR.mkdir(parents=True, exist_ok=True)

    if not INPUT_CSV.exists():
        logging.warning(f"File not found: {INPUT_CSV}")
        return

    with INPUT_CSV.open('r', encoding='utf-8') as f:
        reader = csv.reader(f)
        next(reader, None)
        urls = [row[0] for row in reader if row]

    if not urls:
        logging.info("No URLs found to scrape.")
        return

    driver = setup_optimized_driver()
    wait = WebDriverWait(driver, PAGE_TIMEOUT)

    try:
        with OUTPUT_CSV_FILE.open('w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['Post_URL', 'Full_Content'])

            for i, url in enumerate(urls, 1):
                try:
                    driver.get(url)
                    wait.until(EC.presence_of_element_located((By.TAG_NAME, 'h1')))
                    
                    content = extract_content(driver)
                    writer.writerow([url, content])
                    
                    if i % 10 == 0:
                        logging.info(f"Progress: [{i}/{len(urls)}] scraped.")

                except TimeoutException:
                    logging.error(f"[{i}/{len(urls)}] Timeout loading: {url}")
                except Exception as e:
                    logging.error(f"[{i}/{len(urls)}] Error scraping {url}: {e}")

    finally:
        driver.quit()
        logging.info(f"Scraping completed. Saved to: {OUTPUT_CSV_FILE}")

if __name__ == "__main__":
    main()

2026-03-29 16:47:17,509 - INFO - patching driver executable /home/kongla/.local/share/undetected_chromedriver/undetected_chromedriver
2026-03-29 16:48:11,251 - INFO - Progress: [10/150] scraped.
2026-03-29 16:48:42,479 - INFO - Progress: [20/150] scraped.
2026-03-29 16:49:13,412 - INFO - Progress: [30/150] scraped.
2026-03-29 16:49:40,756 - INFO - Progress: [40/150] scraped.
2026-03-29 16:50:10,231 - INFO - Progress: [50/150] scraped.
2026-03-29 16:50:39,048 - INFO - Progress: [60/150] scraped.
2026-03-29 16:51:08,152 - INFO - Progress: [70/150] scraped.
2026-03-29 16:51:37,652 - INFO - Progress: [80/150] scraped.
2026-03-29 16:52:05,721 - INFO - Progress: [90/150] scraped.
2026-03-29 16:52:33,946 - INFO - Progress: [100/150] scraped.
2026-03-29 16:53:07,045 - INFO - Progress: [110/150] scraped.
2026-03-29 16:53:35,576 - INFO - Progress: [120/150] scraped.
2026-03-29 16:54:01,776 - INFO - Progress: [130/150] scraped.
2026-03-29 16:54:24,709 - INFO - Progress: [140/150] scraped.
2026-03